In [1]:
import tensorflow as tf
import numpy as np

print(tf.__version__)

2.21.0


## Dataset

In [2]:
texts = [
    "i love this movie",
    "this movie is amazing",
    "i hate this movie",
    "this movie is terrible",
    "fantastic film",
    "worst movie ever"
]

labels = [1, 1, 0, 0, 1, 0]

## Text Vectorization

In [4]:
max_tokens = 1000
sequence_length = 6

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

vectorizer.adapt(texts)
print(vectorizer.get_vocabulary())

['', '[UNK]', np.str_('movie'), np.str_('this'), np.str_('is'), np.str_('i'), np.str_('worst'), np.str_('terrible'), np.str_('love'), np.str_('hate'), np.str_('film'), np.str_('fantastic'), np.str_('ever'), np.str_('amazing')]


## Ubah kalimat jadi token

In [5]:
x = vectorizer(texts)

print(x.numpy())

[[ 5  8  3  2  0  0]
 [ 3  2  4 13  0  0]
 [ 5  9  3  2  0  0]
 [ 3  2  4  7  0  0]
 [11 10  0  0  0  0]
 [ 6  2 12  0  0  0]]


## Embedding

In [18]:
embedding_dim = 8

embedding = tf.keras.layers.Embedding(
    input_dim=max_tokens,
    output_dim=embedding_dim
)

embedded = embedding(x)
print(embedded.shape)
print(embedded[0].numpy())

(6, 6, 8)
[[ 0.00534134 -0.01638819 -0.03570003  0.02240782  0.02050931 -0.01035702
  -0.00827892  0.04781972]
 [-0.0016376  -0.00230503 -0.0142463  -0.00336837 -0.02331705 -0.04135305
   0.03754016  0.0274122 ]
 [ 0.02616641 -0.04669724  0.00095568 -0.00735744  0.01165086 -0.04105062
  -0.0360544  -0.03396906]
 [ 0.03300698 -0.01662656 -0.03874863  0.04132037  0.0495795  -0.04712957
   0.04593954  0.02796995]
 [-0.02356276 -0.0201399  -0.04073045 -0.0411207   0.03822866  0.02912294
  -0.01957748 -0.02463877]
 [-0.02356276 -0.0201399  -0.04073045 -0.0411207   0.03822866  0.02912294
  -0.01957748 -0.02463877]]


## Membuat query, key, value

In [20]:
dense_q = tf.keras.layers.Dense(embedding_dim)
dense_k = tf.keras.layers.Dense(embedding_dim)
dense_v = tf.keras.layers.Dense(embedding_dim)

Q = dense_q(embedded)
K = dense_k(embedded)
V = dense_v(embedded)

print(Q.shape)
print(K.shape)
print(V.shape)

(6, 6, 8)
(6, 6, 8)
(6, 6, 8)


## Hitung Attention Score

In [22]:
score = tf.matmul(
    Q,
    K,
    transpose_b=True
)
print(score.shape)

(6, 6, 6)


In [23]:
print(score[0].numpy())

[[ 0.00075038  0.00160755 -0.00124327  0.00158772 -0.00300129 -0.00300129]
 [ 0.00195701  0.00188557 -0.00121941  0.00287135  0.0033289   0.0033289 ]
 [-0.00105616 -0.00087165 -0.00021469 -0.0018455   0.00149987  0.00149987]
 [ 0.00125091  0.00087596  0.00110048  0.00013487  0.00073043  0.00073043]
 [-0.0028223  -0.00151907  0.00423096 -0.00251791 -0.006551   -0.006551  ]
 [-0.0028223  -0.00151907  0.00423096 -0.00251791 -0.006551   -0.006551  ]]


## Scalling (dibagi dengan akar(Dk))

In [24]:
dk = tf.cast(tf.shape(K)[-1], tf.float32)

scaled_score = score / tf.math.sqrt(dk)

## Softmax

In [25]:
attention_weight = tf.nn.softmax(
    scaled_score,
    axis=-1
)
print(attention_weight[0].numpy())

[[0.16674326 0.16679381 0.16662577 0.16679263 0.16652223 0.16652223]
 [0.16666262 0.1666584  0.16647555 0.1667165  0.16674347 0.16674347]
 [0.16661413 0.166625   0.1666637  0.16656764 0.16676477 0.16676477]
 [0.16669302 0.16667092 0.16668415 0.16662726 0.16666235 0.16666235]
 [0.1666547  0.1667315  0.1670708  0.16667263 0.16643514 0.16643514]
 [0.1666547  0.1667315  0.1670708  0.16667263 0.16643514 0.16643514]]


In [26]:
print(tf.reduce_sum(attention_weight[0], axis=-1).numpy())

[0.99999994 1.         1.         1.         0.9999999  0.9999999 ]


## Weighted Sum

In [29]:
output = tf.matmul(
    attention_weight,
    V
)
print(output.shape)
print(output[0].numpy())

(6, 6, 8)
[[-0.00132183  0.00249517 -0.00431303 -0.00253853  0.01141229  0.01783432
  -0.02592802 -0.00754258]
 [-0.001357    0.00248503 -0.00430151 -0.00254048  0.01140429  0.01785709
  -0.025948   -0.00755754]
 [-0.00136997  0.00248059 -0.00430421 -0.00255182  0.01139581  0.01786677
  -0.02595087 -0.00756197]
 [-0.00135274  0.00248698 -0.00430762 -0.002548    0.01140102  0.01785589
  -0.02594077 -0.00755549]
 [-0.00132299  0.00249572 -0.00432232 -0.00255544  0.01140088  0.01783618
  -0.02592586 -0.00754505]
 [-0.00132299  0.00249572 -0.00432232 -0.00255544  0.01140088  0.01783618
  -0.02592586 -0.00754505]]


## Buat layer seflAttention

In [30]:
import tensorflow as tf

class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.Wq = tf.keras.layers.Dense(embed_dim)
        self.Wk = tf.keras.layers.Dense(embed_dim)
        self.Wv = tf.keras.layers.Dense(embed_dim)

    def call(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        score = tf.matmul(Q, K, transpose_b=True)
        dk = tf.cast(tf.shape(K)[-1], tf.float32)
        score = score / tf.math.sqrt(dk)
        attention = tf.nn.softmax(score, axis=-1)
        output = tf.matmul(attention, V)
        return output

In [31]:
attention = SelfAttention(8)
hasil = attention(embedded)
print(hasil.shape)

(6, 6, 8)


## Feed Forward Network (FFN)

In [32]:
class FeedForward(tf.keras.layers.Layer):
    def __init__(self, embed_dim, ff_dim):
        super().__init__()

        self.dense1 = tf.keras.layers.Dense(
            ff_dim,
            activation="relu"
        )

        self.dense2 = tf.keras.layers.Dense(
            embed_dim
        )

    def call(self, x):
        x = self.dense1(x)
        x = self.dense2(x)
        return x

In [33]:
ffn = FeedForward(
    embed_dim=8,
    ff_dim=32
)

hasil = ffn(embedded)

print(hasil.shape)

(6, 6, 8)


## Residual Connection

In [34]:
x = embedded

attention_output = attention(x)

residual = x + attention_output

## Layer Normalization

In [35]:
layer_norm = tf.keras.layers.LayerNormalization()

output = layer_norm(residual)
print(output.shape)

(6, 6, 8)


## FFN + Residual Lagi

In [36]:
ff_output = ffn(output)

output = tf.keras.layers.Add()(
    [output, ff_output]
)

output = tf.keras.layers.LayerNormalization()(output)

## Gabungkan Jadi Encoder

In [ ]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self,
                 embed_dim,
                 ff_dim):
        super().__init__()
        self.attention = SelfAttention(embed_dim)
        self.norm1 = tf.keras.layers.LayerNormalization()
        self.ffn = FeedForward(
            embed_dim,
            ff_dim
        )
        self.norm2 = tf.keras.layers.LayerNormalization()

    def call(self, x):
        attention_output = self.attention(x)
        x = x + attention_output
        x = self.norm1(x)
        ff_output = self.ffn(x)
        x = x + ff_output
        x = self.norm2(x)
        return x

## Coba Encoder

In [38]:
encoder = Encoder(
    embed_dim=8,
    ff_dim=32
)

hasil = encoder(embedded)

print(hasil.shape)

(6, 6, 8)


## Dataset v2

In [39]:
import tensorflow as tf

texts = [
    "i love this movie",
    "this movie is amazing",
    "fantastic film",
    "this is wonderful",

    "i hate this movie",
    "this movie is terrible",
    "worst movie ever",
    "this is awful"
]

labels = [
    1,
    1,
    1,
    1,

    0,
    0,
    0,
    0
]

## Text Vectorization

In [40]:
max_tokens = 1000
sequence_length = 6

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

vectorizer.adapt(texts)

## Embedding

In [41]:
embedding_dim = 32

embedding = tf.keras.layers.Embedding(
    input_dim=max_tokens,
    output_dim=embedding_dim
)

## Encoder

## Banugn Model

In [42]:
inputs = tf.keras.Input(shape=(1,), dtype=tf.string)

x = vectorizer(inputs)
x = embedding(x)

encoder = Encoder(
    embed_dim=32,
    ff_dim=64
)

x = encoder(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dense(32,activation="relu")(x)
outputs = tf.keras.layers.Dense(2,activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

## Compile

In [43]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## Lihat Arsitektur

In [44]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_2            │ (None, 6)              │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (None, 6, 32)          │        32,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_1 (Encoder)             │ (None, 6, 32)          │         7,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,610 (158.63 KB)

 Trainable params: 40,610 (158.63 KB)

 Non-trainable params: 0 (0.00 B)

## Training

In [47]:
x_train = tf.constant(texts, dtype=tf.string)
y_train = tf.constant(labels, dtype=tf.int32)

model.fit(
    x_train,
    y_train,
    epochs=30
)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3750 - loss: 0.9021
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.3750 - loss: 0.7801
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3750 - loss: 0.7296
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6250 - loss: 0.6794
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6250 - loss: 0.6300
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.7500 - loss: 0.5934
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7500 - loss: 0.5624
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7500 - loss: 0.5314
Epoch 9/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8750 - loss: 0.4999
Epoch 10/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 1.0000 - loss: 0.4646
Epoch 11/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.4336
Epoch 12/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 1.0000 - loss: 0.4077
Epo

## Evaluate

In [49]:
loss, acc = model.evaluate(
    x_train,
    y_train
)

print(loss)
print(acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step - accuracy: 1.0000 - loss: 0.0324
0.032378558069467545
1.0


## Predict

In [62]:
test = tf.constant([
    "this movie is fantastic",
    "this movie is terrible",
    "i love this movie",
    "worst movie ever"
], dtype=tf.string)

pred = model.predict(test)

print(pred)
print()

classes = ["Negative", "Positive"]

for kalimat, probabilitas in zip(test.numpy(), pred):

    idx = tf.argmax(probabilitas).numpy()

    print(f"Kalimat : {kalimat.decode()}")
    print(f"Prediksi: {classes[idx]}")
    print(f"Confidence: {probabilitas[idx]*100:.2f}%")
    print("-"*40)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
[[0.04588419 0.9541158 ]
 [0.95917577 0.04082421]
 [0.02828814 0.9717119 ]
 [0.9858542  0.01414574]]

Kalimat : this movie is fantastic
Prediksi: Positive
Confidence: 95.41%
----------------------------------------
Kalimat : this movie is terrible
Prediksi: Negative
Confidence: 95.92%
----------------------------------------
Kalimat : i love this movie
Prediksi: Positive
Confidence: 97.17%
----------------------------------------
Kalimat : worst movie ever
Prediksi: Negative
Confidence: 98.59%
----------------------------------------
